In [ ]:
import os
#os.environ["CUDA_VISIBLE_DEVICES"] = "0"
#os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

#os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"



import Analytic_t_dep_sim as ATDS

import A_nl_altered_sims as A_nl

import Analytic_t_dep_sim_TIDAL_HEAT as ATDS_TH

import Analytic_t_dep_sim_mem_saver as ATDS_MS

import jax
jax.config.update("jax_enable_x64", True)
import numpy as np

import matplotlib.pyplot as plt

# Simulation for Tidal Heating of stellar cluster
## http://arxiv.org/abs/2604.26393
### $R_{1/2}$ defined as median xy plane projected distance between stars in cluster


In [ ]:

import importlib
importlib.reload(ATDS_TH)

# At M22 = 1, r_c = 0.2375 kpc

SphHT = True
integrator = 'leapfrog'
plot = False
dt_override = 20

sim = ATDS_TH.StellarSimTDep(m22 = 1, r_half = 0.02, cluster_R_orbit_kpc = 0.3, cluster_M_Msun=1e4, no_of_particles = 100, no_time_steps = 1000, total_evolve_time = 10, r_min = 0.5, 
                               r_max_enclosing_frac = 0.99, no_radius_bins = 1000, SphHT = SphHT, integrator = integrator, plot = plot, dt_override=dt_override)

In [ ]:
sim.run_simulation()

In [ ]:
positions_all = np.array([p.positions_xyz for p in sim.particles])  # (N_particles, N_steps+1, 3)
r_all         = np.array([p.r_values      for p in sim.particles])  # (N_particles, N_steps+1)
all_vels_cart = np.array([[np.array(v) for v in p.velocities_cart] for p in sim.particles])
kinetic_energy_all = np.array([p.kinetic_energy for p in sim.particles]) # (N_particles, N_steps+1)
potential_energy_all = np.array([p.potential_energy for p in sim.particles]) # (N_particles, N_steps+1)
ang_mom_all = np.array([p.ang_mom for p in sim.particles]) # (N_particles, N_steps+1, 3)

In [ ]:

print(positions_all.shape)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')

for i in range(positions_all.shape[0]):
    ax.scatter(positions_all[i, 0, 0] * sim.u.to_Kpc, positions_all[i, 0, 1] * sim.u.to_Kpc, positions_all[i, 0, 2] * sim.u.to_Kpc, color='blue', label='Initial Position' if i == 0 else "")
    ax.plot(positions_all[i, :, 0] * sim.u.to_Kpc, positions_all[i, :, 1] * sim.u.to_Kpc, positions_all[i, :, 2] * sim.u.to_Kpc)
ax.view_init(elev=0, azim=0)
ax.set_xlabel('X [kpc]')
ax.set_ylabel('Y [kpc]')
ax.set_zlabel('Z [kpc]')
ax.set_xticks([])
plt.show()

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')
for i in range(positions_all.shape[0]):
    ax.scatter(positions_all[i, 0, 0] * sim.u.to_Kpc, positions_all[i, 0, 1] * sim.u.to_Kpc, positions_all[i, 0, 2] * sim.u.to_Kpc, color='blue', label='Initial Position' if i == 0 else "")
    ax.plot(positions_all[i, :, 0] * sim.u.to_Kpc, positions_all[i, :, 1] * sim.u.to_Kpc, positions_all[i, :, 2] * sim.u.to_Kpc)
ax.view_init(elev=90, azim=0)
ax.set_xlabel('X [kpc]')
ax.set_ylabel('Y [kpc]')
ax.set_zlabel('Z [kpc]')
ax.set_zticks([])
plt.show()

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')
for i in range(positions_all.shape[0]):
    ax.scatter(positions_all[i, 0, 0] * sim.u.to_Kpc, positions_all[i, 0, 1] * sim.u.to_Kpc, positions_all[i, 0, 2] * sim.u.to_Kpc, color='blue', label='Initial Position' if i == 0 else "")
    ax.plot(positions_all[i, :, 0] * sim.u.to_Kpc, positions_all[i, :, 1] * sim.u.to_Kpc, positions_all[i, :, 2] * sim.u.to_Kpc)
ax.view_init(elev=0, azim=90)
ax.set_xlabel('X [kpc]')
ax.set_ylabel('Y [kpc]')
ax.set_zlabel('Z [kpc]')
ax.set_yticks([])
plt.show()


In [ ]:
no_time_steps = sim.no_time_steps
# std across particles (axis=0) at each timestep → shape (n_timesteps,)
stellar_v_disp2 = np.sqrt(
    np.std(all_vels_cart[:, :, 0], axis=0)**2 +
    np.std(all_vels_cart[:, :, 1], axis=0)**2 +
    np.std(all_vels_cart[:, :, 2], axis=0)**2
)

com_xy = positions_all[:, :, :2].mean(axis=0)            # (Nsteps, 2) — empirical COM
offsets_xy = positions_all[:, :, :2] - com_xy[None, :, :]
R_half = np.median(np.sqrt(offsets_xy[:, :, 0]**2 + offsets_xy[:, :, 1]**2), axis=0)

com = np.sqrt((positions_all.mean(axis=0)[:,0])**2 + (positions_all.mean(axis=0)[:,1])**2 + (positions_all.mean(axis=0)[:,2])**2)  # (Nsteps, 3) — empirical COM

x = np.arange(no_time_steps + 1)  # Time steps array
plt.plot(x * sim.dt * sim.u.to_Gyr, com * sim.u.to_Kpc)
plt.xlabel('Time [Gyr]')
plt.ylabel('COM Distance from Galactic Center [Kpc]')
plt.title('COM Distance from Galactic Center over Time')
plt.show()

plt.plot(x * sim.dt * sim.u.to_Gyr, stellar_v_disp2 * sim.u.to_kms)
plt.yscale('log')
plt.xlabel('Time [Gyr]')
plt.ylabel('Stellar Velocity Dispersion [km/s]')
plt.title('Stellar Velocity Dispersion over Time')
plt.show()

plt.plot(x * sim.dt * sim.u.to_Gyr, R_half * sim.u.to_Kpc, label='Half-Mass Radius')
#for particle in range(r_all.shape[0]):
    #plt.plot(x * sim.dt * sim.u.to_Gyr, com_xy[particle, :, 0] * sim.u.to_Kpc, alpha = 0.2, color='gray')
plt.axhline(sim.r_half, color='r', linestyle='--', label='Initial Particle Position (r_half)')
plt.xlabel('Time [Gyr]')
plt.yscale('log')
plt.ylabel('Average Stellar Radius [Kpc]')
plt.title('Average Stellar Radius over Time')

# Timescale diagnostics
v0 = np.linalg.norm(all_vels_cart[:, 0, :], axis=1)
mean_T_orb = float(np.mean(2 * np.pi * r_all[:, 0] / v0) * sim.u.to_Gyr)

lambda_db_kpc = 19.15 / (sim.m22 * v0 * sim.u.to_kms)
T_c = lambda_db_kpc / (v0 * sim.u.to_Kpc) * sim.u.to_Gyr

#plt.axhline(np.mean(lambda_db_kpc), color='g', linestyle='--', label='$\\lambda_{{\\rm db}}$')


E = np.array(sim.eigen_energies)
freq_diff = np.abs(E[:, None] - E[None, :])
T_beat = (2 * np.pi / freq_diff) * sim.u.to_Gyr
min_T_beat = np.min(T_beat[np.isfinite(T_beat)])
max_T_beat = np.max(T_beat[np.isfinite(T_beat)])

dt_Gyr = sim.dt * sim.u.to_Gyr

info = (
    f"$T_{{\\rm orb}}$ (mean) = {mean_T_orb:.3f} Gyr\n"
    f"$T_{{\\rm c}}$ = {float(np.mean(T_c)):.3f} Gyr\n"
    f"Beat time band: [{min_T_beat:.3f}, {max_T_beat:.3f}] Gyr\n"
    f"$\\Delta t$ = {dt_Gyr:.4f} Gyr"
)

plt.text(1.02, 0.98, info, transform=plt.gca().transAxes,
         verticalalignment='top', fontsize=9,
         bbox=dict(boxstyle='round', facecolor='paleturquoise', alpha=0.6))

plt.legend(bbox_to_anchor=(1, 0.7))
plt.show()


fig, ax = plt.subplots(1, 2, figsize=(12, 5))

for particle in range(ang_mom_all.shape[0]):
    ax[0].plot(x * sim.dt * sim.u.to_Gyr, (kinetic_energy_all[particle] + potential_energy_all[particle]), label='Total Energy')
ax[0].set_xlabel('Time [Gyr]')
ax[0].set_ylabel('Total Energy [J]')
ax[0].set_title('Total Energy over Time')




for particle in range(ang_mom_all.shape[0]):
    ax[1].plot(x * sim.dt * sim.u.to_Gyr, ang_mom_all[particle], label='Total angular momentum')
ax[1].set_xlabel('Time [Gyr]')
ax[1].set_ylabel('Total angular momentum [kg m^2/s]')
ax[1].set_title('Total angular momentum over Time')


plt.tight_layout()
plt.show()


# Simulation for diffusive heating as in Andrews paper
### Uses Y_lm skip

In [ ]:
import importlib
importlib.reload(ATDS)


SphHT = True
integrator = 'leapfrog'
plot = False
dt_override = 20
ramp_time = 0



sim = ATDS.StellarSimTDep(m22 = 1, r_half = 0.19, r_half_width = 0.05, no_of_particles = 5, no_time_steps = 1000, total_evolve_time = 10, r_min = 20, 
                               r_max_enclosing_frac = 0.99, no_radius_bins = 1000, SphHT = SphHT, integrator = integrator, 
                               plot = plot, dt_override=dt_override, ramp_time=ramp_time)

In [ ]:
sim.run_simulation()

In [ ]:
positions_all = np.array([p.positions_xyz for p in sim.particles])  # (N_particles, N_steps+1, 3)
r_all         = np.array([p.r_values      for p in sim.particles])  # (N_particles, N_steps+1)
all_vels_cart = np.array([[np.array(v) for v in p.velocities_cart] for p in sim.particles])
kinetic_energy_all = np.array([p.kinetic_energy for p in sim.particles]) # (N_particles, N_steps+1)
potential_energy_all = np.array([p.potential_energy for p in sim.particles]) # (N_particles, N_steps+1)
ang_mom_all = np.array([p.ang_mom for p in sim.particles]) # (N_particles, N_steps+1, 3)

In [ ]:

print(positions_all.shape)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')

for i in range(positions_all.shape[0]):
    ax.scatter(positions_all[i, 0, 0] * sim.u.to_Kpc, positions_all[i, 0, 1] * sim.u.to_Kpc, positions_all[i, 0, 2] * sim.u.to_Kpc, color='blue', label='Initial Position' if i == 0 else "")
    ax.plot(positions_all[i, :, 0] * sim.u.to_Kpc, positions_all[i, :, 1] * sim.u.to_Kpc, positions_all[i, :, 2] * sim.u.to_Kpc)
ax.view_init(elev=0, azim=0)
ax.set_xlabel('X [kpc]')
ax.set_ylabel('Y [kpc]')
ax.set_zlabel('Z [kpc]')
ax.set_xticks([])
plt.show()

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')
for i in range(positions_all.shape[0]):
    ax.scatter(positions_all[i, 0, 0] * sim.u.to_Kpc, positions_all[i, 0, 1] * sim.u.to_Kpc, positions_all[i, 0, 2] * sim.u.to_Kpc, color='blue', label='Initial Position' if i == 0 else "")
    ax.plot(positions_all[i, :, 0] * sim.u.to_Kpc, positions_all[i, :, 1] * sim.u.to_Kpc, positions_all[i, :, 2] * sim.u.to_Kpc)
ax.view_init(elev=90, azim=0)
ax.set_xlabel('X [kpc]')
ax.set_ylabel('Y [kpc]')
ax.set_zlabel('Z [kpc]')
ax.set_zticks([])
plt.show()

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')
for i in range(positions_all.shape[0]):
    ax.scatter(positions_all[i, 0, 0] * sim.u.to_Kpc, positions_all[i, 0, 1] * sim.u.to_Kpc, positions_all[i, 0, 2] * sim.u.to_Kpc, color='blue', label='Initial Position' if i == 0 else "")
    ax.plot(positions_all[i, :, 0] * sim.u.to_Kpc, positions_all[i, :, 1] * sim.u.to_Kpc, positions_all[i, :, 2] * sim.u.to_Kpc)
ax.view_init(elev=0, azim=90)
ax.set_xlabel('X [kpc]')
ax.set_ylabel('Y [kpc]')
ax.set_zlabel('Z [kpc]')
ax.set_yticks([])
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import matplotlib 
matplotlib.rcParams['animation.embed_limit'] = 200  
from IPython.display import HTML

N, T, _ = positions_all.shape
dt_Gyr = sim.dt * sim.u.to_Gyr

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')

# Fix axis limits so the view doesn't jump
xyz_min = positions_all.reshape(-1, 3).min(axis=0) * sim.u.to_Kpc
xyz_max = positions_all.reshape(-1, 3).max(axis=0) * sim.u.to_Kpc
ax.set_xlim(xyz_min[0], xyz_max[0])
ax.set_ylim(xyz_min[1], xyz_max[1])
ax.set_zlim(xyz_min[2], xyz_max[2])
ax.set_xlabel('X [kpc]'); ax.set_ylabel('Y [kpc]'); ax.set_zlabel('Z [kpc]')

# One trail line + one head point per particle
trails = [ax.plot([], [], [], lw=0.8)[0] for _ in range(N)]
heads  = [ax.plot([], [], [], 'o', ms=3)[0] for _ in range(N)]

# Time readout
time_text = ax.text2D(0.02, 0.98, '', transform=ax.transAxes,
                      verticalalignment='top', fontsize=11,
                      bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

trail_len = 50  # frames of history to draw; set to T for full trail

def update(frame):
    start = max(0, frame - trail_len)
    for i in range(N):
        seg = positions_all[i, start:frame+1] * sim.u.to_Kpc
        trails[i].set_data(seg[:, 0], seg[:, 1])
        trails[i].set_3d_properties(seg[:, 2])
        p = positions_all[i, frame] * sim.u.to_Kpc
        heads[i].set_data([p[0]], [p[1]])
        heads[i].set_3d_properties([p[2]])
    time_text.set_text(f't = {frame * dt_Gyr:.3f} Gyr')
    return trails + heads + [time_text]

anim = FuncAnimation(fig, update, frames=T, interval=40, blit=False)
HTML(anim.to_jshtml())   
#anim.save('orbits.mp4', fps=25)


In [ ]:
no_time_steps = sim.no_time_steps
# std across particles (axis=0) at each timestep → shape (n_timesteps,)
stellar_v_disp2 = np.sqrt(
    np.std(all_vels_cart[:, :, 0], axis=0)**2 +
    np.std(all_vels_cart[:, :, 1], axis=0)**2 +
    np.std(all_vels_cart[:, :, 2], axis=0)**2
)


x = np.arange(no_time_steps + 1)  # Time steps array

average_r = np.mean(r_all, axis=0)  # Average over particles


plt.plot(x * sim.dt * sim.u.to_Gyr, average_r * sim.u.to_Kpc, label='Average Particle Radius')
for particle in range(r_all.shape[0]-1):
    plt.plot(x * sim.dt * sim.u.to_Gyr, r_all[particle] * sim.u.to_Kpc, alpha = 0.2, color='gray')
plt.plot(x * sim.dt * sim.u.to_Gyr, r_all[-1] * sim.u.to_Kpc, alpha = 0.2, color='gray', label='Individual Particle Radii')
plt.axhline(sim.r_half, color='r', linestyle='--', label='Initial Particle Position (r_half)')

plt.xlabel('Time [Gyr]')
plt.ylabel('Average Stellar Radius [Kpc]')
plt.title('Average Stellar Radius over Time')

# Timescale diagnostics
v0 = np.sqrt(2 * kinetic_energy_all[:, 0])
mean_T_orb = float(np.mean(2 * np.pi * r_all[:, 0] / v0) * sim.u.to_Gyr)

lambda_db_kpc = 19.15 / (sim.m22 * v0 * sim.u.to_kms)
T_c = lambda_db_kpc / (v0 * sim.u.to_Kpc) * sim.u.to_Gyr
plt.axhline(np.mean(lambda_db_kpc), color='g', linestyle='--', label='$\\lambda_{{\\rm db}}$')

E = np.array(sim.eigen_energies)
freq_diff = np.abs(E[:, None] - E[None, :])
T_beat = (2 * np.pi / freq_diff) * sim.u.to_Gyr
min_T_beat = np.min(T_beat[np.isfinite(T_beat)])
max_T_beat = np.max(T_beat[np.isfinite(T_beat)])

dt_Gyr = sim.dt * sim.u.to_Gyr


info = (
    f"$T_{{\\rm orb}}$ (mean) = {mean_T_orb:.3f} Gyr\n"
    f"$T_{{\\rm c}}$ = {float(np.mean(T_c)):.3f} Gyr\n"
    f"Beat time band: [{min_T_beat:.3f}, {max_T_beat:.3f}] Gyr\n"
    f"$\\Delta t$ = {dt_Gyr:.4f} Gyr"
)
plt.text(1.02, 0.98, info, transform=plt.gca().transAxes,
        verticalalignment='top', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='paleturquoise', alpha=0.6))

plt.legend(bbox_to_anchor=(1, 0.7))

plt.show()

fig, ax = plt.subplots(1, 2, figsize=(12, 5))

for particle in range(ang_mom_all.shape[0]):
    ax[0].plot(x * sim.dt * sim.u.to_Gyr, (kinetic_energy_all[particle] + potential_energy_all[particle]), label='Total Energy')
ax[0].set_xlabel('Time [Gyr]')
ax[0].set_ylabel('Total Energy [J]')
ax[0].set_title('Total Energy over Time')




for particle in range(ang_mom_all.shape[0]):
    ax[1].plot(x * sim.dt * sim.u.to_Gyr, ang_mom_all[particle], label='Total angular momentum')
ax[1].set_xlabel('Time [Gyr]')
ax[1].set_ylabel('Total angular momentum [kg m^2/s]')
ax[1].set_title('Total angular momentum over Time')


plt.tight_layout()
plt.show()


# Diffusive heating cprofile run

In [ ]:
import importlib
importlib.reload(ATDS)


SphHT = True
integrator = 'leapfrog'
plot = False
dt_override = 3
ramp_time = 1



sim = ATDS.StellarSimTDep(m22 = 1, r_half = 0.19, r_half_width = 0.1, no_of_particles = 10, no_time_steps = 1000, total_evolve_time = 10, r_min = 20, 
                               r_max_enclosing_frac = 0.99, no_radius_bins = 1000, SphHT = SphHT, integrator = integrator, 
                               plot = plot, dt_override=dt_override, ramp_time=ramp_time)

In [ ]:
sim.run_simulation_profiled(time_output='m22_1_new.prof', memory_output='m22_1_mem_new.txt', top_n=50)


In [ ]:
import os
#os.environ["CUDA_VISIBLE_DEVICES"] = "0"
#os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

#os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import jax
jax.config.update("jax_enable_x64", True)
import numpy as np

import Analytic_t_dep_sim_mem_saver as ATDS_MS


import matplotlib.pyplot as plt

import jax.numpy as jnp

# Running Memory saver version of diffusive heating

In [ ]:
import importlib
importlib.reload(ATDS_MS)


SphHT = True
integrator = 'leapfrog'
plot = False
dt_override = 5
ramp_time = 0

l_band_size = 128
use_multi_gpu = True
sparse_k_batch=8192
r_chunk_size = 128

L_out_frac = 1


sim = ATDS_MS.StellarSimTDep(m22 = 2, r_half = 0.19, r_half_width = 0.05, no_of_particles = 5, no_time_steps = 1000, total_evolve_time = 10, r_min = 20, 
                               r_max_enclosing_frac = 0.99, no_radius_bins = 1000, SphHT = SphHT, integrator = integrator, 
                               plot = plot, dt_override=dt_override, ramp_time=ramp_time, l_band_size=l_band_size, use_multi_gpu = use_multi_gpu,
                                 sparse_k_batch=sparse_k_batch, r_chunk_size=r_chunk_size, compute_dtype=jnp.complex128, L_out_frac=L_out_frac)

In [ ]:
sim.run_simulation() 

In [ ]:
max_rho_00 = np.array(sim.maximum_rho_00)
x = np.arange(sim.no_time_steps + 1)  # Time steps array


plt.plot(x * sim.dt * sim.u.to_Gyr, max_rho_00 * sim.u.to_Msun / (sim.u.to_Kpc)**3)
plt.xlabel('Time (Gyr)')
plt.ylabel('Max |rho_00| (Msun/kpc^3)')
plt.title(f'Max |rho_00| over Time steps for m22 = {sim.m22}')
plt.yscale('log')
plt.show()


In [ ]:
positions_all = np.array([p.positions_xyz for p in sim.particles])  # (N_particles, N_steps+1, 3)
r_all         = np.array([p.r_values      for p in sim.particles])  # (N_particles, N_steps+1)
all_vels_cart = np.array([[np.array(v) for v in p.velocities_cart] for p in sim.particles])
kinetic_energy_all = np.array([p.kinetic_energy for p in sim.particles]) # (N_particles, N_steps+1)
potential_energy_all = np.array([p.potential_energy for p in sim.particles]) # (N_particles, N_steps+1)
ang_mom_all = np.array([p.ang_mom for p in sim.particles]) # (N_particles, N_steps+1, 3)

In [ ]:

print(positions_all.shape)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')

for i in range(positions_all.shape[0]):
    ax.scatter(positions_all[i, 0, 0] * sim.u.to_Kpc, positions_all[i, 0, 1] * sim.u.to_Kpc, positions_all[i, 0, 2] * sim.u.to_Kpc, color='blue', label='Initial Position' if i == 0 else "")
    ax.plot(positions_all[i, :, 0] * sim.u.to_Kpc, positions_all[i, :, 1] * sim.u.to_Kpc, positions_all[i, :, 2] * sim.u.to_Kpc)
ax.view_init(elev=0, azim=0)
ax.set_xlabel('X [kpc]')
ax.set_ylabel('Y [kpc]')
ax.set_zlabel('Z [kpc]')
ax.set_xticks([])
plt.show()

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')
for i in range(positions_all.shape[0]):
    ax.scatter(positions_all[i, 0, 0] * sim.u.to_Kpc, positions_all[i, 0, 1] * sim.u.to_Kpc, positions_all[i, 0, 2] * sim.u.to_Kpc, color='blue', label='Initial Position' if i == 0 else "")
    ax.plot(positions_all[i, :, 0] * sim.u.to_Kpc, positions_all[i, :, 1] * sim.u.to_Kpc, positions_all[i, :, 2] * sim.u.to_Kpc)
ax.view_init(elev=90, azim=0)
ax.set_xlabel('X [kpc]')
ax.set_ylabel('Y [kpc]')
ax.set_zlabel('Z [kpc]')
ax.set_zticks([])
plt.show()

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')
for i in range(positions_all.shape[0]):
    ax.scatter(positions_all[i, 0, 0] * sim.u.to_Kpc, positions_all[i, 0, 1] * sim.u.to_Kpc, positions_all[i, 0, 2] * sim.u.to_Kpc, color='blue', label='Initial Position' if i == 0 else "")
    ax.plot(positions_all[i, :, 0] * sim.u.to_Kpc, positions_all[i, :, 1] * sim.u.to_Kpc, positions_all[i, :, 2] * sim.u.to_Kpc)
ax.view_init(elev=0, azim=90)
ax.set_xlabel('X [kpc]')
ax.set_ylabel('Y [kpc]')
ax.set_zlabel('Z [kpc]')
ax.set_yticks([])
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import matplotlib 
matplotlib.rcParams['animation.embed_limit'] = 200  
from IPython.display import HTML

N, T, _ = positions_all.shape
dt_Gyr = sim.dt * sim.u.to_Gyr

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')

# Fix axis limits so the view doesn't jump
xyz_min = positions_all.reshape(-1, 3).min(axis=0) * sim.u.to_Kpc 
xyz_max = positions_all.reshape(-1, 3).max(axis=0) * sim.u.to_Kpc 
ax.set_xlim(xyz_min[0], xyz_max[0])
ax.set_ylim(xyz_min[1], xyz_max[1])
ax.set_zlim(xyz_min[2], xyz_max[2])
ax.set_xlabel('X [kpc]'); ax.set_ylabel('Y [kpc]'); ax.set_zlabel('Z [kpc]')

# One trail line + one head point per particle
trails = [ax.plot([], [], [], lw=0.8)[0] for _ in range(N)]
heads  = [ax.plot([], [], [], 'o', ms=3)[0] for _ in range(N)]

# Time readout
time_text = ax.text2D(0.02, 0.98, '', transform=ax.transAxes,
                      verticalalignment='top', fontsize=11,
                      bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

trail_len = 50  # frames of history to draw; set to T for full trail

def update(frame):
    start = max(0, frame - trail_len)
    for i in range(N):
        seg = positions_all[i, start:frame+1] * sim.u.to_Kpc
        trails[i].set_data(seg[:, 0], seg[:, 1])
        trails[i].set_3d_properties(seg[:, 2])
        p = positions_all[i, frame] * sim.u.to_Kpc
        heads[i].set_data([p[0]], [p[1]])
        heads[i].set_3d_properties([p[2]])
    time_text.set_text(f't = {frame * dt_Gyr:.3f} Gyr')
    return trails + heads + [time_text]

anim = FuncAnimation(fig, update, frames=T, interval=40, blit=False)
HTML(anim.to_jshtml())   
#anim.save('orbits.mp4', fps=25)


In [ ]:
import numpy as np, matplotlib.pyplot as plt

dt_Gyr = sim.dt * sim.u.to_Gyr
n_ramp = sim.n_ramp_steps
N, T, _ = positions_all.shape

r_kpc  = r_all * sim.u.to_Kpc
E_tot  = kinetic_energy_all + potential_energy_all
speed  = np.linalg.norm(all_vels_cart, axis=2) * sim.u.to_kms
theta  = np.degrees(np.arccos(np.clip(positions_all[:,:,2]/np.clip(r_all,1e-30,None), -1, 1)))

print(f"ramp ends at step {n_ramp} (t={n_ramp*dt_Gyr:.2f} Gyr); final radii [kpc]: {np.round(r_kpc[:,-1],2)}\n")
ejected = np.where(r_kpc[:,-1] > 5*r_kpc[:,0])[0]

for i in ejected:
    runaway = int(np.argmax(r_kpc[i] > 2*r_kpc[i,0]))
    dspeed  = np.abs(np.diff(speed[i])); js = int(np.argmax(dspeed))
    dE      = np.abs(np.diff(E_tot[i])); jE = int(np.argmax(dE))
    print(f"--- particle {i}:  r {r_kpc[i,0]:.3f} -> {r_kpc[i,-1]:.1f} kpc")
    print(f"  runaway starts step {runaway} (t={runaway*dt_Gyr:.2f} Gyr, during ramp: {runaway<n_ramp})")
    print(f"  biggest speed jump {dspeed[js]:.2f} km/s at step {js}, theta there = {theta[i,js]:.1f} deg")
    print(f"  theta range over run: {theta[i].min():.1f} - {theta[i].max():.1f} deg")
    print(f"  biggest single-step |dE| at step {jE}: {dE[jE]:.2e}  (median {np.median(dE):.2e}, ratio {dE[jE]/np.median(dE):.0f})\n")

fig, ax = plt.subplots(1, 3, figsize=(18,4))
x = np.arange(T) * dt_Gyr
for i in range(N):
    c = 'red' if i in ejected else 'gray'
    ax[0].plot(x, r_kpc[i], c=c); ax[1].plot(x, speed[i], c=c); ax[2].plot(x, E_tot[i], c=c)
for a, lbl in zip(ax, ['r [kpc]','speed [km/s]','E_tot']):
    a.axvline(n_ramp*dt_Gyr, ls='--', c='blue'); a.set_xlabel('t [Gyr]'); a.set_ylabel(lbl)
ax[0].set_yscale('log')
plt.tight_layout(); plt.show()


In [ ]:
no_time_steps = sim.no_time_steps
# std across particles (axis=0) at each timestep → shape (n_timesteps,)
stellar_v_disp2 = np.sqrt(
    np.std(all_vels_cart[:, :, 0], axis=0)**2 +
    np.std(all_vels_cart[:, :, 1], axis=0)**2 +
    np.std(all_vels_cart[:, :, 2], axis=0)**2
)


x = np.arange(no_time_steps + 1)  # Time steps array


plt.plot(x * sim.dt * sim.u.to_Gyr, stellar_v_disp2 * sim.u.to_kms)
plt.xlabel('Time [Gyr]')
plt.ylabel('Stellar Velocity Dispersion [km/s]')
plt.title('Stellar Velocity Dispersion over Time')
plt.show()

R_half = np.mean(r_all, axis=0)  # (N_steps+1,) median across particles at each timestep

plt.plot(x * sim.dt * sim.u.to_Gyr, R_half * sim.u.to_Kpc, label='Average Particle Radius')
for particle in range(r_all.shape[0]-1):
    plt.plot(x * sim.dt * sim.u.to_Gyr, r_all[particle, :] * sim.u.to_Kpc, alpha = 0.2, color='gray')
plt.plot(x * sim.dt * sim.u.to_Gyr, r_all[-1, :] * sim.u.to_Kpc, alpha = 0.2, color='gray', label='Individual Particle Radii')
init_r_half = R_half[0] * sim.u.to_Kpc
plt.axhline(init_r_half, color='r', linestyle='--', label='Initial Mean Radius')
plt.xlabel('Time [Gyr]')
plt.ylabel('Average Stellar Radius [Kpc]')
plt.title('Average Stellar Radius over Time')

# Timescale diagnostics
v0 = np.linalg.norm(all_vels_cart[:, 0, :], axis=1)
mean_T_orb = float(np.mean(2 * np.pi * r_all[:, 0] / v0) * sim.u.to_Gyr)

lambda_db_kpc = 19.15 / (sim.m22 * v0 * sim.u.to_kms)
T_c = lambda_db_kpc / (v0 * sim.u.to_Kpc) * sim.u.to_Gyr

#plt.axhline(np.mean(lambda_db_kpc), color='g', linestyle='--', label='$\\lambda_{{\\rm db}}$')


E = np.array(sim.eigen_energies)
freq_diff = np.abs(E[:, None] - E[None, :])
T_beat = (2 * np.pi / freq_diff) * sim.u.to_Gyr
min_T_beat = np.min(T_beat[np.isfinite(T_beat)])
max_T_beat = np.max(T_beat[np.isfinite(T_beat)])

dt_Gyr = sim.dt * sim.u.to_Gyr

info = (
    f"$T_{{\\rm orb}}$ (mean) = {mean_T_orb:.3f} Gyr\n"
    f"$T_{{\\rm c}}$ = {float(np.mean(T_c)):.3f} Gyr\n"
    f"Beat time band: [{min_T_beat:.3f}, {max_T_beat:.3f}] Gyr\n"
    f"$\\Delta t$ = {dt_Gyr:.4f} Gyr"
)

plt.text(1.02, 0.98, info, transform=plt.gca().transAxes,
         verticalalignment='top', fontsize=9,
         bbox=dict(boxstyle='round', facecolor='paleturquoise', alpha=0.6))

plt.legend(bbox_to_anchor=(1, 0.7))
plt.show()


fig, ax = plt.subplots(1, 2, figsize=(12, 5))

for particle in range(ang_mom_all.shape[0]):
    ax[0].plot(x * sim.dt * sim.u.to_Gyr, (kinetic_energy_all[particle] + potential_energy_all[particle]), label='Total Energy')
ax[0].set_xlabel('Time [Gyr]')
ax[0].set_ylabel('Total Energy [J]')
ax[0].set_title('Total Energy over Time')




for particle in range(ang_mom_all.shape[0]):
    ax[1].plot(x * sim.dt * sim.u.to_Gyr, ang_mom_all[particle], label='Total angular momentum')
ax[1].set_xlabel('Time [Gyr]')
ax[1].set_ylabel('Total angular momentum [kg m^2/s]')
ax[1].set_title('Total angular momentum over Time')


plt.tight_layout()
plt.show()


# Running memory saver with truncation of L_max_out to see if makes a difference

In [ ]:
import os
#os.environ["CUDA_VISIBLE_DEVICES"] = "0"
#os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

#os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import jax
jax.config.update("jax_enable_x64", True)
import numpy as np

import Analytic_t_dep_sim_mem_saver as ATDS_MS


import matplotlib.pyplot as plt

import jax.numpy as jnp

from time import time

: 

In [ ]:
import importlib
importlib.reload(ATDS_MS)


SphHT = True
integrator = 'leapfrog'
plot = False
dt_override = 20
ramp_time = 0

l_band_size = 128
use_multi_gpu = True
sparse_k_batch=8192
r_chunk_size = 128

L_sims = []
times = []

for L_out_frac in [1, 0.9, 0.8, 0.7, 0.6, 0.5]:

  print(f"Running sim with L_out_frac = {L_out_frac}...")

  sim = ATDS_MS.StellarSimTDep(m22 = 2, r_half = 0.19, r_half_width = 0.05, no_of_particles = 5, no_time_steps = 1000, total_evolve_time = 10, r_min = 20, 
                                r_max_enclosing_frac = 0.99, no_radius_bins = 1000, SphHT = SphHT, integrator = integrator, 
                                plot = plot, dt_override=dt_override, ramp_time=ramp_time, l_band_size=l_band_size, use_multi_gpu = use_multi_gpu,
                                  sparse_k_batch=sparse_k_batch, r_chunk_size=r_chunk_size, compute_dtype=jnp.complex128, L_out_frac=L_out_frac)

  time_start = time()

  sim.run_simulation() 

  time_end = time()
  elapsed = time_end - time_start
  times.append(elapsed)
  
  L_sims.append(sim)

In [ ]:
for time in times:
    print(f"Sim with L_out_frac = {L_out_frac} took {time:.2f} seconds")

In [ ]:
positions_all = []
r_all = []
all_vels_cart = []
kinetic_energy_all = []
potential_energy_all = []
ang_mom_all = []

for sim in L_sims:

    positions_all.append(np.array([p.positions_xyz for p in sim.particles]))  # (N_particles, N_steps+1, 3)
    r_all.append(np.array([p.r_values      for p in sim.particles]))  # (N_particles, N_steps+1)
    all_vels_cart.append(np.array([[np.array(v) for v in p.velocities_cart] for p in sim.particles]))
    kinetic_energy_all.append(np.array([p.kinetic_energy for p in sim.particles])) # (N_particles, N_steps+1)
    potential_energy_all.append(np.array([p.potential_energy for p in sim.particles])) # (N_particles, N_steps+1)
    ang_mom_all.append(np.array([p.ang_mom for p in sim.particles])) # (N_particles, N_steps+1, 3)

In [ ]:
no_time_steps = L_sims[0].no_time_steps


x = np.arange(no_time_steps + 1)  # Time steps array

for i, sim in enumerate(L_sims):

    R_half = np.mean(r_all[i], axis=0)  # (N_steps+1,) 

    plt.plot(x * sim.dt * sim.u.to_Gyr, R_half * sim.u.to_Kpc, label=f'Half-Mass Radius (L_out_frac={sim.L_out_frac})', alpha=0.7)
    plt.xlabel('Time [Gyr]')
    plt.ylabel('Average Stellar Radius [Kpc]')
    plt.title('Average Stellar Radius over Time')

    # Timescale diagnostics
    v0 = np.linalg.norm(all_vels_cart[i][:, 0, :], axis=1)
    mean_T_orb = float(np.mean(2 * np.pi * r_all[i][:, 0] / v0) * sim.u.to_Gyr)

    lambda_db_kpc = 19.15 / (sim.m22 * v0 * sim.u.to_kms)
    T_c = lambda_db_kpc / (v0 * sim.u.to_Kpc) * sim.u.to_Gyr

    #plt.axhline(np.mean(lambda_db_kpc), color='g', linestyle='--', label='$\\lambda_{{\\rm db}}$')


    E = np.array(sim.eigen_energies)
    freq_diff = np.abs(E[:, None] - E[None, :])
    T_beat = (2 * np.pi / freq_diff) * sim.u.to_Gyr
    min_T_beat = np.min(T_beat[np.isfinite(T_beat)])
    max_T_beat = np.max(T_beat[np.isfinite(T_beat)])

    dt_Gyr = sim.dt * sim.u.to_Gyr

    info = (
        f"$T_{{\\rm orb}}$ (mean) = {mean_T_orb:.3f} Gyr\n"
        f"$T_{{\\rm c}}$ = {float(np.mean(T_c)):.3f} Gyr\n"
        f"Beat time band: [{min_T_beat:.3f}, {max_T_beat:.3f}] Gyr\n"
        f"$\\Delta t$ = {dt_Gyr:.4f} Gyr"
    )

    plt.text(1.02, 0.98 - i * 0.2, info, transform=plt.gca().transAxes,
            verticalalignment='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='paleturquoise', alpha=0.6))


    
plt.axhline(L_sims[0].r_half, color='r', linestyle='--', label='Initial Particle Position (r_half)')
    
plt.legend(bbox_to_anchor=(1.6, 0.7))
plt.show()


R_half_truth = np.mean(r_all[0], axis=0)  # L_out_frac = 1
errors = []
for i, sim in enumerate(L_sims):
    R_half = np.mean(r_all[i], axis=0)
    error = np.abs(R_half - R_half_truth) / R_half_truth
    errors.append(error)


plt.figure(figsize=(10, 6))
for i, sim in enumerate(L_sims):
    plt.plot(x * sim.dt * sim.u.to_Gyr, errors[i], label=f'L_out_frac={sim.L_out_frac}', alpha=0.7)
plt.xlabel('Time [Gyr]')
plt.ylabel('Relative Error in Half-Mass Radius')
plt.title('Relative Error in Half-Mass Radius over Time')
plt.yscale('log')
plt.legend()
plt.show()

# Running memory saver with getting rid of low a_nl modes


In [ ]:
import os
#os.environ["CUDA_VISIBLE_DEVICES"] = "0"
#os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

#os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"



import Analytic_t_dep_sim_mem_saver_a_nl as ATDS_MS_anl

import jax
jax.config.update("jax_enable_x64", True)
import numpy as np

import matplotlib.pyplot as plt

import jax.numpy as jnp

In [ ]:
import importlib
importlib.reload(ATDS_MS_anl)


SphHT = True
integrator = 'leapfrog'
plot = False
dt_override = 20
ramp_time = 1

l_band_size = 128
use_multi_gpu = True
sparse_k_batch=8192
r_chunk_size = 128

a_j_threshold = 1e-7


sim2 = ATDS_MS_anl.StellarSimTDep(m22 = 2, r_half = 0.19, r_half_width = 0.05, no_of_particles = 5, no_time_steps = 1000, total_evolve_time = 10, r_min = 20, 
                               r_max_enclosing_frac = 0.99, no_radius_bins = 1000, SphHT = SphHT, integrator = integrator, 
                               plot = plot, dt_override=dt_override, ramp_time=ramp_time, l_band_size=l_band_size, use_multi_gpu = use_multi_gpu,
                                 sparse_k_batch=sparse_k_batch, r_chunk_size=r_chunk_size, compute_dtype=jnp.complex128, a_j_threshold = a_j_threshold)

In [ ]:
sim2.run_simulation()

In [ ]:
positions_all = np.array([p.positions_xyz for p in sim2.particles])  # (N_particles, N_steps+1, 3)
r_all         = np.array([p.r_values      for p in sim2.particles])  # (N_particles, N_steps+1)
all_vels_cart = np.array([[np.array(v) for v in p.velocities_cart] for p in sim2.particles])
kinetic_energy_all = np.array([p.kinetic_energy for p in sim2.particles]) # (N_particles, N_steps+1)
potential_energy_all = np.array([p.potential_energy for p in sim2.particles]) # (N_particles, N_steps+1)
ang_mom_all = np.array([p.ang_mom for p in sim2.particles]) # (N_particles, N_steps+1, 3)

In [ ]:

print(positions_all.shape)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')

for i in range(positions_all.shape[0]):
    ax.scatter(positions_all[i, 0, 0] * sim2.u.to_Kpc, positions_all[i, 0, 1] * sim2.u.to_Kpc, positions_all[i, 0, 2] * sim2.u.to_Kpc, color='blue', label='Initial Position' if i == 0 else "")
    ax.plot(positions_all[i, :, 0] * sim2.u.to_Kpc, positions_all[i, :, 1] * sim2.u.to_Kpc, positions_all[i, :, 2] * sim2.u.to_Kpc)
ax.view_init(elev=0, azim=0)
ax.set_xlabel('X [kpc]')
ax.set_ylabel('Y [kpc]')
ax.set_zlabel('Z [kpc]')
ax.set_xticks([])
plt.show()

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')
for i in range(positions_all.shape[0]):
    ax.scatter(positions_all[i, 0, 0] * sim2.u.to_Kpc, positions_all[i, 0, 1] * sim2.u.to_Kpc, positions_all[i, 0, 2] * sim2.u.to_Kpc, color='blue', label='Initial Position' if i == 0 else "")
    ax.plot(positions_all[i, :, 0] * sim2.u.to_Kpc, positions_all[i, :, 1] * sim2.u.to_Kpc, positions_all[i, :, 2] * sim2.u.to_Kpc)
ax.view_init(elev=90, azim=0)
ax.set_xlabel('X [kpc]')
ax.set_ylabel('Y [kpc]')
ax.set_zlabel('Z [kpc]')
ax.set_zticks([])
plt.show()

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')
for i in range(positions_all.shape[0]):
    ax.scatter(positions_all[i, 0, 0] * sim2.u.to_Kpc, positions_all[i, 0, 1] * sim2.u.to_Kpc, positions_all[i, 0, 2] * sim2.u.to_Kpc, color='blue', label='Initial Position' if i == 0 else "")
    ax.plot(positions_all[i, :, 0] * sim2.u.to_Kpc, positions_all[i, :, 1] * sim2.u.to_Kpc, positions_all[i, :, 2] * sim2.u.to_Kpc)
ax.view_init(elev=0, azim=90)
ax.set_xlabel('X [kpc]')
ax.set_ylabel('Y [kpc]')
ax.set_zlabel('Z [kpc]')
ax.set_yticks([])
plt.show()


In [ ]:
# Difference between first and second sim

positions_all_1 = np.array([p.positions_xyz for p in sim.particles])  # (N_particles, N_steps+1, 3)
positions_all_2 = np.array([p.positions_xyz for p in sim2.particles])

diff = np.linalg.norm(positions_all_1 - positions_all_2, axis=2)  # (N_particles, N_steps+1)
fig = plt.figure(figsize=(10, 6))
for i in range(diff.shape[0]):
    plt.plot(np.arange(diff.shape[1]) * sim.dt * sim.u.to_Gyr, diff[i], label=f'Particle {i}')
plt.xlabel('Time [Gyr]')
plt.ylabel('Position Difference [kpc]')
plt.title('Position Difference Between Simulations Over Time')
plt.legend()
plt.show()
    

# Altering the original $a_{nl}$ coefficients so states with larger (n,l) have higher $a_{nl}$
### First altering n,l states which correspond to periods of oscillation that are close to the orbital period of test masses
### Keeping sum of a_nl un-normalised so the period of the orbits dont change after chainging the $a_{nl}$

In [ ]:

import importlib
importlib.reload(A_nl)

animate = False
SphHT = False
integrator = 'leapfrog'
a_nl_range = 'orbital'
plot = True

sim = A_nl.StellarSimTDep(m22 = 1, r_half = 0.19, no_of_particles = 5, no_time_steps = 1000, total_evolve_time = 10, r_min = 20, 
                               r_max_enclosing_frac = 0.99, no_radius_bins = 1000, SphHT = SphHT, integrator = integrator, a_nl_range= a_nl_range, plot = plot, boost_factor=1e3, animate=animate, animate_every=10)

In [ ]:
sim.run_simulation()

In [ ]:
positions_all = np.array([p.positions_xyz for p in sim.particles])  # (N_particles, N_steps+1, 3)
r_all         = np.array([p.r_values      for p in sim.particles])  # (N_particles, N_steps+1)
v_disp_all    = np.array([p.stellar_v_disp for p in sim.particles]) # (N_particles, N_steps+1)
kinetic_energy_all = np.array([p.kinetic_energy for p in sim.particles]) # (N_particles, N_steps+1)
potential_energy_all = np.array([p.potential_energy for p in sim.particles]) # (N_particles, N_steps+1)
ang_mom_all = np.array([p.ang_mom for p in sim.particles]) # (N_particles, N_steps+1, 3)

In [ ]:
print(positions_all.shape)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')

for i in range(positions_all.shape[0]):
    ax.scatter(positions_all[i, 0, 0], positions_all[i, 0, 1], positions_all[i, 0, 2], color='blue', label='Initial Position' if i == 0 else "")
    ax.plot(positions_all[i, :, 0], positions_all[i, :, 1], positions_all[i, :, 2])
ax.view_init(elev=0, azim=0)
ax.set_xlabel('X [kpc]')
ax.set_ylabel('Y [kpc]')
ax.set_zlabel('Z [kpc]')
ax.set_xticks([])
plt.show()

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')
for i in range(positions_all.shape[0]):
    ax.scatter(positions_all[i, 0, 0], positions_all[i, 0, 1], positions_all[i, 0, 2], color='blue', label='Initial Position' if i == 0 else "")
    ax.plot(positions_all[i, :, 0], positions_all[i, :, 1], positions_all[i, :, 2])
ax.view_init(elev=90, azim=0)
ax.set_xlabel('X [kpc]')
ax.set_ylabel('Y [kpc]')
ax.set_zlabel('Z [kpc]')
ax.set_zticks([])
plt.show()

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')
for i in range(positions_all.shape[0]):
    ax.scatter(positions_all[i, 0, 0], positions_all[i, 0, 1], positions_all[i, 0, 2], color='blue', label='Initial Position' if i == 0 else "")
    ax.plot(positions_all[i, :, 0], positions_all[i, :, 1], positions_all[i, :, 2])
ax.view_init(elev=0, azim=90)
ax.set_xlabel('X [kpc]')
ax.set_ylabel('Y [kpc]')
ax.set_zlabel('Z [kpc]')
ax.set_yticks([])
plt.show()


In [ ]:
time_step2 = sim.time_step
stellar_v_disp2 = np.mean(v_disp_all, axis=0)  # Average over particles

average_r2 = np.mean(r_all, axis=0)  # Average over particles


x = np.linspace(0, time_step2, len(stellar_v_disp2))

plt.plot(x * sim.dt * sim.u.to_Gyr, stellar_v_disp2 * sim.u.to_kms)
plt.yscale('log')
plt.xlabel('Time [Gyr]')
plt.ylabel('Stellar Velocity Dispersion [km/s]')
plt.title('Stellar Velocity Dispersion over Time')
plt.show()

plt.plot(x * sim.dt * sim.u.to_Gyr, average_r2 * sim.u.to_Kpc, label='Average Particle Radius')
for particle in range(r_all.shape[0]):
    plt.plot(x * sim.dt * sim.u.to_Gyr, r_all[particle] * sim.u.to_Kpc, alpha = 0.2, color='gray')
plt.axhline(sim.r_half, color='r', linestyle='--', label='Initial Particle Position (r_half)')
plt.xlabel('Time [Gyr]')
plt.ylabel('Average Stellar Radius [Kpc]')
plt.title('Average Stellar Radius over Time')

# Timescale diagnostics
v0 = np.sqrt(2 * kinetic_energy_all[:, 0])
mean_T_orb = float(np.mean(2 * np.pi * r_all[:, 0] / v0) * sim.u.to_Gyr)

lambda_db_kpc = 19.15 / (sim.m22 * v0 * sim.u.to_kms)
T_c = lambda_db_kpc / (v0 * sim.u.to_Kpc) * sim.u.to_Gyr


E = np.array(sim.eigen_energies)
freq_diff = np.abs(E[:, None] - E[None, :])
T_beat = (2 * np.pi / freq_diff) * sim.u.to_Gyr
min_T_beat = np.min(T_beat[np.isfinite(T_beat)])
max_T_beat = np.max(T_beat[np.isfinite(T_beat)])

dt_Gyr = sim.dt * sim.u.to_Gyr


info = (
    f"$T_{{\\rm orb}}$ (mean) = {mean_T_orb:.3f} Gyr\n"
    f"$T_{{\\rm c}}$ = {float(T_c[0]):.3f} Gyr\n"
    f"Beat time band: [{min_T_beat:.3f}, {max_T_beat:.3f}] Gyr\n"
    f"$\\Delta t$ = {dt_Gyr:.4f} Gyr"
)
plt.text(0.02, 0.98, info, transform=plt.gca().transAxes,
         verticalalignment='top', fontsize=9,
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.6))

plt.legend(loc = 'lower right')
plt.show()



fig, ax = plt.subplots(1, 2, figsize=(12, 5))

for particle in range(ang_mom_all.shape[0]):
    ax[0].plot(x * sim.dt * sim.u.to_Gyr, (kinetic_energy_all[particle] + potential_energy_all[particle]), label='Total Energy')
ax[0].set_xlabel('Time [Gyr]')
ax[0].set_ylabel('Total Energy [J]')
ax[0].set_title('Total Energy over Time')




for particle in range(ang_mom_all.shape[0]):
    ax[1].plot(x * sim.dt * sim.u.to_Gyr, ang_mom_all[particle], label='Total angular momentum')
ax[1].set_xlabel('Time [Gyr]')
ax[1].set_ylabel('Total angular momentum [kg m^2/s]')
ax[1].set_title('Total angular momentum over Time')


plt.tight_layout()
plt.show()


## Altering large period $a_{nl}$ coeffs

In [ ]:

import importlib
importlib.reload(A_nl)

animate = False
SphHT = False
integrator = 'leapfrog'
a_nl_range = 'large'
plot = True

sim = A_nl.StellarSimTDep(m22 = 1, r_half = 0.19, no_of_particles = 5, no_time_steps = 500, total_evolve_time = 10, r_min = 20, 
                               r_max_enclosing_frac = 0.99, no_radius_bins = 1000, SphHT = SphHT, integrator = integrator, a_nl_range= a_nl_range, plot = plot, boost_factor=1e5, animate=animate, animate_every=10)

In [ ]:
sim.run_simulation()

In [ ]:
positions_all = np.array([p.positions_xyz for p in sim.particles])  # (N_particles, N_steps+1, 3)
r_all         = np.array([p.r_values      for p in sim.particles])  # (N_particles, N_steps+1)
v_disp_all    = np.array([p.stellar_v_disp for p in sim.particles]) # (N_particles, N_steps+1)
kinetic_energy_all = np.array([p.kinetic_energy for p in sim.particles]) # (N_particles, N_steps+1)
potential_energy_all = np.array([p.potential_energy for p in sim.particles]) # (N_particles, N_steps+1)
ang_mom_all = np.array([p.ang_mom for p in sim.particles]) # (N_particles, N_steps+1, 3)

In [ ]:
print(positions_all.shape)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')

for i in range(positions_all.shape[0]):
    ax.scatter(positions_all[i, 0, 0], positions_all[i, 0, 1], positions_all[i, 0, 2], color='blue', label='Initial Position' if i == 0 else "")
    ax.plot(positions_all[i, :, 0], positions_all[i, :, 1], positions_all[i, :, 2])
ax.view_init(elev=0, azim=0)
ax.set_xlabel('X [kpc]')
ax.set_ylabel('Y [kpc]')
ax.set_zlabel('Z [kpc]')
ax.set_xticks([])
plt.show()

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')
for i in range(positions_all.shape[0]):
    ax.scatter(positions_all[i, 0, 0], positions_all[i, 0, 1], positions_all[i, 0, 2], color='blue', label='Initial Position' if i == 0 else "")
    ax.plot(positions_all[i, :, 0], positions_all[i, :, 1], positions_all[i, :, 2])
ax.view_init(elev=90, azim=0)
ax.set_xlabel('X [kpc]')
ax.set_ylabel('Y [kpc]')
ax.set_zlabel('Z [kpc]')
ax.set_zticks([])
plt.show()

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')
for i in range(positions_all.shape[0]):
    ax.scatter(positions_all[i, 0, 0], positions_all[i, 0, 1], positions_all[i, 0, 2], color='blue', label='Initial Position' if i == 0 else "")
    ax.plot(positions_all[i, :, 0], positions_all[i, :, 1], positions_all[i, :, 2])
ax.view_init(elev=0, azim=90)
ax.set_xlabel('X [kpc]')
ax.set_ylabel('Y [kpc]')
ax.set_zlabel('Z [kpc]')
ax.set_yticks([])
plt.show()


In [ ]:
time_step2 = sim.time_step
stellar_v_disp2 = np.mean(v_disp_all, axis=0)  # Average over particles

average_r2 = np.mean(r_all, axis=0)  # Average over particles


x = np.linspace(0, time_step2, len(stellar_v_disp2))

plt.plot(x * sim.dt * sim.u.to_Gyr, stellar_v_disp2 * sim.u.to_kms)
plt.yscale('log')
plt.xlabel('Time [Gyr]')
plt.ylabel('Stellar Velocity Dispersion [km/s]')
plt.title('Stellar Velocity Dispersion over Time')
plt.show()

plt.plot(x * sim.dt * sim.u.to_Gyr, average_r2 * sim.u.to_Kpc, label='Average Particle Radius')
for particle in range(r_all.shape[0]):
    plt.plot(x * sim.dt * sim.u.to_Gyr, r_all[particle] * sim.u.to_Kpc, alpha = 0.2, color='gray')
plt.axhline(sim.r_half, color='r', linestyle='--', label='Initial Particle Position (r_half)')
plt.xlabel('Time [Gyr]')
plt.ylabel('Average Stellar Radius [Kpc]')
plt.title('Average Stellar Radius over Time')

# Timescale diagnostics
v0 = np.sqrt(2 * kinetic_energy_all[:, 0])
mean_T_orb = float(np.mean(2 * np.pi * r_all[:, 0] / v0) * sim.u.to_Gyr)

lambda_db_kpc = 19.15 / (sim.m22 * v0 * sim.u.to_kms)
T_c = lambda_db_kpc / (v0 * sim.u.to_Kpc) * sim.u.to_Gyr


E = np.array(sim.eigen_energies)
freq_diff = np.abs(E[:, None] - E[None, :])
T_beat = (2 * np.pi / freq_diff) * sim.u.to_Gyr
min_T_beat = np.min(T_beat[np.isfinite(T_beat)])
max_T_beat = np.max(T_beat[np.isfinite(T_beat)])

dt_Gyr = sim.dt * sim.u.to_Gyr


info = (
    f"$T_{{\\rm orb}}$ (mean) = {mean_T_orb:.3f} Gyr\n"
    f"$T_{{\\rm c}}$ = {float(T_c[0]):.3f} Gyr\n"
    f"Beat time band: [{min_T_beat:.3f}, {max_T_beat:.3f}] Gyr\n"
    f"$\\Delta t$ = {dt_Gyr:.4f} Gyr"
)
plt.text(0.02, 0.98, info, transform=plt.gca().transAxes,
         verticalalignment='top', fontsize=9,
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.6))

plt.legend(loc = 'lower right')
plt.show()


fig, ax = plt.subplots(1, 2, figsize=(12, 5))

for particle in range(ang_mom_all.shape[0]):
    ax[0].plot(x * sim.dt * sim.u.to_Gyr, (kinetic_energy_all[particle] + potential_energy_all[particle]), label='Total Energy')
ax[0].set_xlabel('Time [Gyr]')
ax[0].set_ylabel('Total Energy [J]')
ax[0].set_title('Total Energy over Time')




for particle in range(ang_mom_all.shape[0]):
    ax[1].plot(x * sim.dt * sim.u.to_Gyr, ang_mom_all[particle], label='Total angular momentum')
ax[1].set_xlabel('Time [Gyr]')
ax[1].set_ylabel('Total angular momentum [kg m^2/s]')
ax[1].set_title('Total angular momentum over Time')


plt.tight_layout()
plt.show()


## Altering small period $a_{nl}$ coeffs

In [ ]:

import importlib
importlib.reload(A_nl)

animate = False
SphHT = False
integrator = 'leapfrog'
a_nl_range = 'small'
plot = True

sim = A_nl.StellarSimTDep(m22 = 1, r_half = 0.19, no_of_particles = 5, no_time_steps = 500, total_evolve_time = 10, r_min = 20, 
                               r_max_enclosing_frac = 0.99, no_radius_bins = 1000, SphHT = SphHT, integrator = integrator, a_nl_range= a_nl_range, plot = plot, boost_factor=1e6, animate=animate, animate_every=10)

In [ ]:
sim.run_simulation()

In [ ]:
positions_all = np.array([p.positions_xyz for p in sim.particles])  # (N_particles, N_steps+1, 3)
r_all         = np.array([p.r_values      for p in sim.particles])  # (N_particles, N_steps+1)
v_disp_all    = np.array([p.stellar_v_disp for p in sim.particles]) # (N_particles, N_steps+1)
kinetic_energy_all = np.array([p.kinetic_energy for p in sim.particles]) # (N_particles, N_steps+1)
potential_energy_all = np.array([p.potential_energy for p in sim.particles]) # (N_particles, N_steps+1)
ang_mom_all = np.array([p.ang_mom for p in sim.particles]) # (N_particles, N_steps+1, 3)

In [ ]:
print(positions_all.shape)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')

for i in range(positions_all.shape[0]):
    ax.scatter(positions_all[i, 0, 0], positions_all[i, 0, 1], positions_all[i, 0, 2], color='blue', label='Initial Position' if i == 0 else "")
    ax.plot(positions_all[i, :, 0], positions_all[i, :, 1], positions_all[i, :, 2])
ax.view_init(elev=0, azim=0)
ax.set_xlabel('X [kpc]')
ax.set_ylabel('Y [kpc]')
ax.set_zlabel('Z [kpc]')
ax.set_xticks([])
plt.show()

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')
for i in range(positions_all.shape[0]):
    ax.scatter(positions_all[i, 0, 0], positions_all[i, 0, 1], positions_all[i, 0, 2], color='blue', label='Initial Position' if i == 0 else "")
    ax.plot(positions_all[i, :, 0], positions_all[i, :, 1], positions_all[i, :, 2])
ax.view_init(elev=90, azim=0)
ax.set_xlabel('X [kpc]')
ax.set_ylabel('Y [kpc]')
ax.set_zlabel('Z [kpc]')
ax.set_zticks([])
plt.show()

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(0, 0, 0, color='red', label='Galactic Center')
for i in range(positions_all.shape[0]):
    ax.scatter(positions_all[i, 0, 0], positions_all[i, 0, 1], positions_all[i, 0, 2], color='blue', label='Initial Position' if i == 0 else "")
    ax.plot(positions_all[i, :, 0], positions_all[i, :, 1], positions_all[i, :, 2])
ax.view_init(elev=0, azim=90)
ax.set_xlabel('X [kpc]')
ax.set_ylabel('Y [kpc]')
ax.set_zlabel('Z [kpc]')
ax.set_yticks([])
plt.show()


In [ ]:
time_step2 = sim.time_step
stellar_v_disp2 = np.mean(v_disp_all, axis=0)  # Average over particles

average_r2 = np.mean(r_all, axis=0)  # Average over particles


x = np.linspace(0, time_step2, len(stellar_v_disp2))

plt.plot(x * sim.dt * sim.u.to_Gyr, stellar_v_disp2 * sim.u.to_kms)
plt.yscale('log')
plt.xlabel('Time [Gyr]')
plt.ylabel('Stellar Velocity Dispersion [km/s]')
plt.title('Stellar Velocity Dispersion over Time')
plt.show()

plt.plot(x * sim.dt * sim.u.to_Gyr, average_r2 * sim.u.to_Kpc, label='Average Particle Radius')
for particle in range(r_all.shape[0]):
    plt.plot(x * sim.dt * sim.u.to_Gyr, r_all[particle] * sim.u.to_Kpc, alpha = 0.2, color='gray')
plt.axhline(sim.r_half, color='r', linestyle='--', label='Initial Particle Position (r_half)')
plt.xlabel('Time [Gyr]')
plt.ylabel('Average Stellar Radius [Kpc]')
plt.title('Average Stellar Radius over Time')

# Timescale diagnostics
v0 = np.sqrt(2 * kinetic_energy_all[:, 0])
mean_T_orb = float(np.mean(2 * np.pi * r_all[:, 0] / v0) * sim.u.to_Gyr)

lambda_db_kpc = 19.15 / (sim.m22 * v0 * sim.u.to_kms)
T_c = lambda_db_kpc / (v0 * sim.u.to_Kpc) * sim.u.to_Gyr


E = np.array(sim.eigen_energies)
freq_diff = np.abs(E[:, None] - E[None, :])
T_beat = (2 * np.pi / freq_diff) * sim.u.to_Gyr
min_T_beat = np.min(T_beat[np.isfinite(T_beat)])
max_T_beat = np.max(T_beat[np.isfinite(T_beat)])

dt_Gyr = sim.dt * sim.u.to_Gyr


info = (
    f"$T_{{\\rm orb}}$ (mean) = {mean_T_orb:.3f} Gyr\n"
    f"$T_{{\\rm c}}$ = {float(T_c[0]):.3f} Gyr\n"
    f"Beat time band: [{min_T_beat:.3f}, {max_T_beat:.3f}] Gyr\n"
    f"$\\Delta t$ = {dt_Gyr:.4f} Gyr"
)
plt.text(0.02, 0.98, info, transform=plt.gca().transAxes,
         verticalalignment='top', fontsize=9,
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.6))

plt.legend(loc = 'lower right')
plt.show()


fig, ax = plt.subplots(1, 2, figsize=(12, 5))

for particle in range(ang_mom_all.shape[0]):
    ax[0].plot(x * sim.dt * sim.u.to_Gyr, (kinetic_energy_all[particle] + potential_energy_all[particle]), label='Total Energy')
ax[0].set_xlabel('Time [Gyr]')
ax[0].set_ylabel('Total Energy [J]')
ax[0].set_title('Total Energy over Time')




for particle in range(ang_mom_all.shape[0]):
    ax[1].plot(x * sim.dt * sim.u.to_Gyr, ang_mom_all[particle], label='Total angular momentum')
ax[1].set_xlabel('Time [Gyr]')
ax[1].set_ylabel('Total angular momentum [kg m^2/s]')
ax[1].set_title('Total angular momentum over Time')


plt.tight_layout()
plt.show()
